In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import sys
sys.path.append('/content/drive/MyDrive/ColabNotebooks/proyecto_aa2')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

print(os.path.exists("/content/drive/MyDrive/ColabNotebooks/proyecto_aa2"))

True


In [ ]:
import torch
import numpy as np

from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader

from torch.optim import Adam

from score_model import ScoreNet
import diffusion_process as dfp
from tqdm.notebook import tqdm

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cpu


In [ ]:
batch_size = 128

data = datasets.MNIST(
    root="data",
    train=True,
    download=True,
    transform=ToTensor()
)

data_loader = DataLoader(
    data,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2
)

100%|██████████| 9.91M/9.91M [00:01<00:00, 5.84MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 154kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.49MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 8.80MB/s]


In [ ]:
def beta_t(t): #linear
    beta_min = 0.1
    beta_max = 20.0
    return beta_min + t * (beta_max - beta_min)

def drift(x_t, t):
    beta = beta_t(t)
    return -0.5 * beta[:, None, None, None] * x_t

def diffusion(t):
    beta = beta_t(t)
    return torch.sqrt(beta)

def mu_t(x_0, t):
    beta_min = 0.1
    beta_max = 20.0

    integral = beta_min * t + 0.5 * (beta_max - beta_min) * t**2
    coeff = torch.exp(-0.5 * integral)

    return coeff[:, None, None, None] * x_0
def sigma_t(t):
    beta_min = 0.1
    beta_max = 20.0

    integral = beta_min * t + 0.5 * (beta_max - beta_min) * t**2
    return torch.sqrt(1 - torch.exp(-integral))

In [ ]:
diffusion_process = dfp.GaussianDiffussionProcess(
    drift,
    diffusion,
    mu_t,
    sigma_t
)

In [ ]:
from functools import partial

score_model = ScoreNet(
    marginal_prob_std=partial(sigma_t)
)

score_model = score_model.to(device)

In [ ]:
optimizer = Adam(score_model.parameters(), lr=1e-4)
n_epochs = 35

for epoch in range(n_epochs):

    avg_loss = 0.0
    num_items = 0

    for x, _ in tqdm(data_loader, desc=f"Epoch {epoch}"):

        x = x.to(device)


        loss = diffusion_process.loss_function(score_model, x)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        avg_loss += loss.item() * x.shape[0]
        num_items += x.shape[0]

    print(f"Epoch {epoch} | Loss: {avg_loss / num_items:.4f}")


    #torch.save(score_model.state_dict(), f"model_epoch_{epoch}.pth")

torch.save(score_model.state_dict(), "final_model_vp_linear.pth")

Epoch 0:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 0 | Loss: 594.0881


Epoch 1:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 1 | Loss: 145.8279


Epoch 2:   0%|          | 0/469 [00:00<?, ?it/s]

...

Epoch 32 | Loss: 21.5814


Epoch 33:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 33 | Loss: 20.9359


Epoch 34:   0%|          | 0/469 [00:00<?, ?it/s]

Epoch 34 | Loss: 21.0241
